# 📘 Notebook 01: Learning by Trial and Error

### Course: *From Random Moves to Winning Agents: Reinforcement Learning with Games*
**Instructor:** Ioannis Tsioulis

*Module A: Foundations · Notebook 1 of 4 in this module · (1 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Describe reinforcement learning with its four words: **agent**, **environment**, **action**, **reward**
- Explain how it differs from supervised learning
- Estimate the value of an action from experience with an **incremental average**
- Explain the **exploration versus exploitation** trade-off and why a purely greedy agent fails
- Implement **epsilon-greedy** and **UCB** action selection and compare them with a measurement
- Read a learning curve averaged over many runs, and explain why the averaging is necessary

> **Prerequisites:** Python (functions, loops, lists) and a first contact with NumPy arrays and matplotlib. If these are new, see *From Python to Deep Learning & Fine-Tuning*, Notebooks 05 and 07.
>
> 🔭 **Why this notebook matters later:** every agent in this course, from the snake of Notebook 07 to the lander of Notebook 14, faces the question we meet here in its simplest form: use what I already know, or try something new?

> ℹ️ **Setup note.** This notebook needs only NumPy and matplotlib, which Google Colab already has. Every cell finishes within a few seconds.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. What is reinforcement learning?

### Intuition first
Nobody taught you to ride a bicycle by showing you ten thousand labelled examples of correct pedalling. You tried, wobbled, fell, tried again, and your body gradually kept what worked. A child learning that a stove is hot, a dog learning to sit for a biscuit, a player getting better at a video game: all of them learn from the **consequences** of what they do.

**Reinforcement learning (RL)** is the part of machine learning that works this way. A program is not told which action is correct. It acts, receives a number that says how good the outcome was, and has to work out for itself how to get more of it.

### Formal definition
Four words describe every problem in this course:

| Word | Meaning | Bicycle example |
|---|---|---|
| **Agent** | the learner and decision maker | you |
| **Environment** | everything the agent interacts with | the bicycle, the road, gravity |
| **Action** | what the agent chooses to do | lean, steer, pedal |
| **Reward** | a number sent back after each action | staying upright is good, falling is bad |

The agent's goal is always the same: collect as much **total reward** as possible.

### Why it matters
In supervised learning (the regression and classification of other courses) every training example comes with the right answer. Here there is no answer sheet, which changes three things:

- The agent only learns about the actions it **actually tries**. An action never tried is an action never understood.
- The feedback **evaluates** an action ("that was worth 3"). It does not **instruct** ("you should have done B").
- From Notebook 02 on, an action will also change the situation the agent finds itself in next.

This notebook isolates the first two points by removing the third. There is one situation, repeated for ever, and a handful of actions.

## 2. A row of slot machines

### Intuition first
You walk into an arcade with ten slot machines. Each pays out a different amount on average, and you do not know which is which. You have 1,000 coins. How do you play?

If you spend all the coins on the first machine, you never find out whether the others are better. If you spread them evenly, you waste most of them on bad machines. This is the **multi-armed bandit** problem (a slot machine is a "one-armed bandit"), and it is the purest form of the dilemma between trying things and using what you know.

### Formal definition
A **k-armed bandit** has `k` actions. Action `a` has a hidden **true value** $q_*(a)$, its average reward. Each time the agent plays `a`, it receives that average plus random noise.

In [ ]:
class Bandit:
    """k slot machines. The true average pay-out of each one is hidden from the agent."""

    def __init__(self, k=10, seed=0):
        self.rng = np.random.default_rng(seed)
        self.true_values = self.rng.normal(0, 1, k)      # the secret
        self.k = k

    def pull(self, arm):
        return float(self.true_values[arm] + self.rng.normal(0, 1))   # average plus noise


bandit = Bandit()
print("Five pulls of arm 3:", [round(bandit.pull(3), 2) for i in range(5)])
print("Five more:           ", [round(bandit.pull(3), 2) for i in range(5)])

Look at the numbers. The pulls of one arm differ a lot from each other, so a single pull tells you very little. That noise is what makes the problem hard.

We, the authors of the experiment, are allowed to look at the secret:

In [ ]:
print("True values:", np.round(bandit.true_values, 2))
print("Best arm:   ", bandit.true_values.argmax())

The agent never sees `true_values`. It only sees the rewards of the arms it pulls.

## 3. Estimating how good an action is

### Intuition first
To judge a restaurant you average your past visits. After each new visit you do not recompute the whole average from your diary: you nudge your opinion a little towards the latest experience.

### Formal definition
The agent keeps an **estimate** $Q(a)$ for every action: the average of the rewards received from `a` so far. With $N(a)$ the number of times `a` was tried, the average can be updated after each new reward $R$ without storing the old ones:

$$Q(a) \leftarrow Q(a) + \frac{1}{N(a)}\,\big(R - Q(a)\big)$$

In words: **new estimate = old estimate + step size × (what happened − what I expected)**. The bracket is the **error** of the old estimate. If the reward was higher than expected, the estimate goes up, and the other way round.

### Why it matters
Keep this pattern in mind. Almost every learning rule in this course, including the ones that train neural networks to land a spacecraft, has exactly this shape.

In [ ]:
bandit = Bandit()
arm = 3
Q, N = 0.0, 0

for pull in range(1, 1001):
    reward = bandit.pull(arm)
    N += 1
    Q += (reward - Q) / N                     # the incremental average
    if pull in (1, 10, 100, 1000):
        print(f"after {pull:>4} pulls: estimate {Q:6.3f}")

print(f"true value:        {bandit.true_values[arm]:6.3f}")

The estimate wanders at first and settles on the true value as the evidence accumulates.

## 4. The greedy agent and its problem

### Intuition first
The obvious strategy is: always play the machine with the best estimate so far. This is called acting **greedily**. It sounds sensible and it has a flaw. Suppose the truly best machine pays badly on your first try, by bad luck. Its estimate is now low, the greedy agent never touches it again, and so the estimate is never corrected.

### Formal definition
- **Exploiting** means choosing the action with the highest current estimate: $\arg\max_a Q(a)$.
- **Exploring** means choosing something else, to improve the estimates.
- An **epsilon-greedy** agent exploits most of the time, and with a small probability $\varepsilon$ (epsilon) picks an action completely at random.

With $\varepsilon = 0$ the agent is purely greedy. With $\varepsilon = 0.1$ it explores on one step in ten.

In [ ]:
def play(bandit, epsilon, steps=1000, seed=0):
    """One agent plays one bandit. Returns the rewards and the arms it chose."""
    rng = np.random.default_rng(seed)
    Q = np.zeros(bandit.k)                    # estimates
    N = np.zeros(bandit.k)                    # how often each arm was tried
    rewards, arms = [], []
    for t in range(steps):
        if rng.random() < epsilon:
            arm = int(rng.integers(bandit.k))             # explore
        else:
            arm = int(Q.argmax())                         # exploit
        reward = bandit.pull(arm)
        N[arm] += 1
        Q[arm] += (reward - Q[arm]) / N[arm]
        rewards.append(reward)
        arms.append(arm)
    return np.array(rewards), np.array(arms)


for epsilon in (0.0, 0.1):
    bandit = Bandit(seed=0)
    rewards, arms = play(bandit, epsilon)
    best = bandit.true_values.argmax()
    print(f"epsilon = {epsilon}:  average reward {rewards.mean():.2f},  "
          f"favourite arm {np.bincount(arms).argmax()} (best is {best}),  "
          f"chose the best arm {100 * (arms == best).mean():.0f}% of the time")

In this particular game the greedy agent was lucky. It met the best arm early, stayed with it, and beat the agent that kept exploring. So was the warning above wrong?

One game proves little, because both the bandit and the agent involve chance. A different seed could reverse the result. In reinforcement learning a **single run is an anecdote**. To compare strategies fairly we repeat the experiment on many different bandits and average.

Running 500 separate Python loops would be slow, so the next function plays 500 bandits **at the same time**: every variable gets one row per bandit. The logic is the same as in `play`.

In [ ]:
def experiment(choose, steps=1000, runs=500, k=10, seed=0):
    """Average behaviour of a strategy over many random bandits, all played in parallel."""
    rng = np.random.default_rng(seed)
    true_values = rng.normal(0, 1, (runs, k))
    best = true_values.argmax(axis=1)
    rows = np.arange(runs)
    Q = np.zeros((runs, k))
    N = np.zeros((runs, k))
    average_reward = np.zeros(steps)
    optimal = np.zeros(steps)
    for t in range(steps):
        arms = choose(Q, N, t, rng)                        # one arm per bandit
        rewards = true_values[rows, arms] + rng.normal(0, 1, runs)
        N[rows, arms] += 1
        Q[rows, arms] += (rewards - Q[rows, arms]) / N[rows, arms]
        average_reward[t] = rewards.mean()
        optimal[t] = (arms == best).mean()
    return average_reward, optimal


def epsilon_greedy(epsilon):
    def choose(Q, N, t, rng):
        greedy = Q.argmax(axis=1)
        random_arms = rng.integers(0, Q.shape[1], len(Q))
        explore = rng.random(len(Q)) < epsilon
        return np.where(explore, random_arms, greedy)
    return choose

In [ ]:
results = {f"epsilon = {e}": experiment(epsilon_greedy(e)) for e in (0.0, 0.01, 0.1)}

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for label, (reward, optimal) in results.items():
    axes[0].plot(reward, label=label)
    axes[1].plot(100 * optimal, label=label)
axes[0].set_title("Average reward (500 bandits)")
axes[0].set_xlabel("step"); axes[0].set_ylabel("reward"); axes[0].grid(True); axes[0].legend()
axes[1].set_title("How often the best arm was chosen")
axes[1].set_xlabel("step"); axes[1].set_ylabel("% of bandits"); axes[1].grid(True); axes[1].legend()
plt.show()

for label, (reward, optimal) in results.items():
    print(f"{label:15s} average reward over the last 200 steps: {reward[-200:].mean():.2f},  "
          f"best arm chosen: {100 * optimal[-200:].mean():.0f}%")

Read the two plots together.

- The **greedy** agent (epsilon 0) improves fastest at the very start and then stops improving. In most of the 500 games it has locked on to an arm that is good but not the best.
- **Epsilon 0.1** finds the best arm in most games. Its curve flattens too, for a different reason: it keeps choosing at random on one step in ten, even when it no longer needs to.
- **Epsilon 0.01** learns slowly, and is still climbing when the experiment ends.

> ⚠️ **Common pitfalls**
>
> - **Judging a strategy from one run.** Change `seed` in the single game above and watch the conclusion change. Always average.
> - **Thinking exploration is wasted effort.** The reward lost while exploring is the price of information. The greedy agent pays nothing and ends up worse.
> - **Starting every estimate at 0 without thinking.** The starting values are a choice, and Exercise 3 shows that they change the behaviour.

## 5. Exploring with a purpose: UCB

### Intuition first
Epsilon-greedy explores blindly: when it explores, it is as likely to pick an arm it has tried 300 times as one it has tried twice. A curious person does better. They give the benefit of the doubt to what they know least about: "I have only been to that restaurant once. It might be much better than my one visit suggests."

### Formal definition
**Upper Confidence Bound (UCB)** selection adds a bonus for uncertainty to each estimate and then acts greedily on the sum:

$$A_t = \arg\max_a \left[\, Q(a) + c\,\sqrt{\frac{\ln t}{N(a)}} \,\right]$$

The bonus is large when $N(a)$ is small (the arm has rarely been tried) and shrinks every time the arm is played. The constant $c$ sets how curious the agent is. An arm that has never been tried counts as infinitely interesting and is played first.

In [ ]:
def ucb(c=2.0):
    def choose(Q, N, t, rng):
        bonus = c * np.sqrt(np.log(t + 1) / np.maximum(N, 1e-9))   # a huge bonus for untried arms
        return (Q + bonus).argmax(axis=1)
    return choose


results["UCB, c = 2"] = experiment(ucb(2.0))

plt.figure(figsize=(7, 4))
for label in ("epsilon = 0.1", "UCB, c = 2"):
    plt.plot(results[label][0], label=label)
plt.title("Average reward: blind versus directed exploration")
plt.xlabel("step"); plt.ylabel("reward"); plt.grid(True); plt.legend()
plt.show()

for label in ("epsilon = 0.1", "UCB, c = 2"):
    reward, optimal = results[label]
    print(f"{label:15s} total reward per game: {reward.sum():7.1f},  "
          f"average over the last 200 steps: {reward[-200:].mean():.2f}")

UCB collects more reward on this problem. It spends its exploration where the uncertainty is, and as the counts grow it explores less and less by itself, with no schedule to tune.

> 🧠 UCB needs a count $N(a)$ for every action. That is easy with ten slot machines. When the "situation" becomes a picture of a game screen, counting visits is no longer possible, which is why most agents later in this course fall back on epsilon-greedy.

---
## ✏️ Exercises

The exercises reuse `Bandit`, `play`, `experiment`, `epsilon_greedy` and `ucb` from above.

### Exercise 1 (The update rule by hand)
An arm has been pulled three times and gave the rewards 4, 0 and 5. Without NumPy, apply the incremental rule $Q \leftarrow Q + (R - Q)/N$ three times, starting from `Q = 0`, and print `Q` after each step. Check that the final value equals the ordinary average.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
Q, N = 0.0, 0
for reward in [4, 0, 5]:
    N += 1
    Q += (reward - Q) / N
    print(f"after reward {reward}: Q = {Q:.3f}")

print("ordinary average:", (4 + 0 + 5) / 3)
```

*The incremental rule is the average, computed without remembering the past rewards.*
</details>

### Exercise 2 (How much exploration?)
Run `experiment` for epsilon = 0, 0.01, 0.05, 0.1, 0.2 and 0.5 and print the average reward per step over the whole game for each. Which value is best over 1,000 steps? Why is 0.5 poor even though it certainly finds the best arm?

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
for epsilon in (0, 0.01, 0.05, 0.1, 0.2, 0.5):
    reward, optimal = experiment(epsilon_greedy(epsilon))
    print(f"epsilon = {epsilon:<5} average reward per step: {reward.mean():.3f}")
```

*With epsilon 0.5 the agent knows the best arm early but plays at random half of the time, so it throws its knowledge away. Finding the best action and using it are two different things.*
</details>

### Exercise 3 (Optimism)
Change nothing in the strategy, only the starting point: write a copy of `experiment` in which the estimates start at `+5` for every arm instead of 0, and run it with a **purely greedy** agent (epsilon 0). Compare its average reward over the last 200 steps with the same agent starting from 0. Explain why starting high makes a greedy agent explore.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def experiment_optimistic(choose, start, steps=1000, runs=500, k=10, seed=0):
    rng = np.random.default_rng(seed)
    true_values = rng.normal(0, 1, (runs, k))
    rows = np.arange(runs)
    Q = np.full((runs, k), float(start))          # the only change
    N = np.zeros((runs, k))
    average_reward = np.zeros(steps)
    for t in range(steps):
        arms = choose(Q, N, t, rng)
        rewards = true_values[rows, arms] + rng.normal(0, 1, runs)
        N[rows, arms] += 1
        Q[rows, arms] += 0.1 * (rewards - Q[rows, arms])     # constant step, so the start is forgotten slowly
        average_reward[t] = rewards.mean()
    return average_reward

for start in (0, 5):
    reward = experiment_optimistic(epsilon_greedy(0.0), start)
    print(f"greedy, estimates start at {start}: average reward over the last 200 steps {reward[-200:].mean():.2f}")
```

*Every real reward is a disappointment compared with +5, so the estimate of whatever arm was just played drops below the others, and the greedy agent moves on to the next one. It tries everything several times before it settles. The solution uses a constant step size of 0.1: with the `1/N` step the first reward would erase the optimistic start completely.*
</details>

### Exercise 4 (A decaying epsilon)
Exploration is most useful early. Write a strategy `decaying_epsilon` in which epsilon at step `t` is `1 / (1 + t / 20)`: it starts at 1 and falls towards 0. Compare its total reward per game with epsilon = 0.1.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def decaying_epsilon(Q, N, t, rng):
    epsilon = 1 / (1 + t / 20)
    greedy = Q.argmax(axis=1)
    random_arms = rng.integers(0, Q.shape[1], len(Q))
    return np.where(rng.random(len(Q)) < epsilon, random_arms, greedy)

for label, choose in (("epsilon = 0.1", epsilon_greedy(0.1)), ("decaying epsilon", decaying_epsilon)):
    reward, optimal = experiment(choose)
    print(f"{label:17s} total reward per game {reward.sum():7.1f},  best arm at the end {100 * optimal[-200:].mean():.0f}%")
```

*A schedule that explores a lot at first and little later is the standard choice in the rest of the course.*
</details>

### Exercise 5 (Two adverts)
A website shows one of two adverts to each visitor. Advert A is clicked by 4% of visitors and advert B by 5%, but nobody knows this. The reward is 1 for a click and 0 otherwise. Simulate 20,000 visitors with an epsilon-greedy agent (epsilon 0.1) and report how many times each advert was shown and the estimated click rates. Why is this problem harder than it looks?

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
rng = np.random.default_rng(1)
click_rate = np.array([0.04, 0.05])            # hidden from the agent
Q, N = np.zeros(2), np.zeros(2)

for visitor in range(20000):
    advert = int(rng.integers(2)) if rng.random() < 0.1 else int(Q.argmax())
    reward = float(rng.random() < click_rate[advert])
    N[advert] += 1
    Q[advert] += (reward - Q[advert]) / N[advert]

for name, shown, estimate in zip("AB", N, Q):
    print(f"advert {name}: shown {int(shown):>6} times, estimated click rate {100 * estimate:.2f}%")
```

*The two click rates are close and the rewards are almost always 0, so thousands of visitors are needed before the difference shows through the noise. Run it with other seeds: sometimes the agent favours the wrong advert for a long time.*
</details>

### Exercise 6 (A world that changes (a little harder))
Real pay-outs drift. Write an experiment with **one** 10-armed bandit in which, after every step, each true value moves by a small random amount (`rng.normal(0, 0.01)`). Run 10,000 steps with epsilon = 0.1 twice: once with the sample-average step `1/N`, once with a constant step size of 0.1. Print the average reward over the last 2,000 steps for both, averaged over 100 different seeds. Which is better, and why?

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def drifting(step_size, steps=10000, seed=0):
    rng = np.random.default_rng(seed)
    true_values = np.zeros(10)
    Q, N = np.zeros(10), np.zeros(10)
    total = 0.0
    for t in range(steps):
        arm = int(rng.integers(10)) if rng.random() < 0.1 else int(Q.argmax())
        reward = true_values[arm] + rng.normal()
        N[arm] += 1
        step = 1 / N[arm] if step_size is None else step_size
        Q[arm] += step * (reward - Q[arm])
        if t >= steps - 2000:
            total += reward
        true_values += rng.normal(0, 0.01, 10)             # the world drifts
    return total / 2000

for label, step_size in (("sample average (1/N)", None), ("constant step 0.1", 0.1)):
    scores = [drifting(step_size, seed=s) for s in range(100)]
    print(f"{label:22s} average reward at the end: {np.mean(scores):.2f}")
```

*The sample average gives every past reward the same weight, so old information that is no longer true holds the estimate back. A constant step size forgets the distant past and keeps following the change. Constant step sizes are the norm from Notebook 05 onwards.*
</details>

## 🔑 Key takeaways

- Reinforcement learning is learning from **reward**: an agent acts in an environment and tries to collect as much total reward as possible.
- The feedback **evaluates** the action taken. It never says what the best action would have been.
- An action's value is estimated by **new = old + step × (reward − old)**, a pattern that returns throughout the course.
- A **greedy** agent can lock on to a poor action for ever. Some **exploration** is necessary.
- **Epsilon-greedy** explores at random; **UCB** explores what it is least sure about.
- Results in RL are noisy. Compare methods on **averages over many runs**, never on one.

---
**Next:** *Notebook 02: Worlds, States and Rewards*, where actions start to have consequences that last.

---
*© Ioannis Tsioulis. *From Random Moves to Winning Agents: Reinforcement Learning with Games*. Prepared for educational use.*